[![Open in nbviewer](https://img.shields.io/badge/Open%20in-nbviewer-orange?logo=jupyter)](https://nbviewer.org/github/eacooper/NymeriaGazeTools/blob/main/examples/quick_analysis.ipynb)

> **Charts are interactive on nbviewer** — zoom, pan, and hover over data points. On GitHub, charts appear blank due to JavaScript restrictions.

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
from pathlib import Path
import nymeria_gaze_tools as ngt

DATA_ROOT = Path("../data/processed")

In [3]:
# Load the catalog and explore what's available
catalog = ngt.load_metadata(data_root=DATA_ROOT)

print("Participants:", ngt.list_participants(catalog)[:5], "...")
print("Activities: ", ngt.list_activities(catalog))
print("Locations:  ", ngt.list_locations(catalog)[:5], "...")

Participants: ['adriana_gonzalez', 'aimee_davis', 'alan_burns', 'albert_hammond', 'alec_meza'] ...
Activities:  ['S1-Relax_at_home', 'S10-Housekeeping', 'S11-Laundary', 'S12-Game_night', 'S13-Charades', 'S14-By_my_desk', 'S15-Do_as_I_command', 'S16-Simon_says', 'S17-In_the_office', 'S18-Hike', 'S19-Fresh_air', 'S2-Where_is_X', 'S20-Party', 'S3-Welcome_to_my_place', 'S4-Body_stretch', 'S5-Workout', 'S6-Dance', 'S7-Cooking', 'S8-Having_a_meal', 'S9-Making_a_mess']
Locations:   ['Loc_04', 'Loc_05', 'Loc_06', 'Loc_07', 'Loc_08'] ...


In [4]:
# Filter to sessions of interest
sessions = ngt.filter_sessions(catalog, script="S7-Cooking", has_gaze_data=True)
print(f"{len(sessions)} sessions found")
sessions.head()

154 sessions found


,sequence_uid,date,session_id,fake_name,act_id,location,script,participant_gender,participant_height_cm,participant_weight_kg,participant_bmi,participant_age_group,participant_ethnicity,gaze_type,has_gaze_data
0,20230628_s0_adriana_gonzalez_act4_rzxlmo,20230628,s0,adriana_gonzalez,act4,Loc_10,S7-Cooking,Male,177.0,81.0,25.90,25-30,Caucasian,personalized,True
1,20230929_s1_alan_burns_act4_zch0c7,20230929,s1,alan_burns,act4,Loc_29,S7-Cooking,Female,151.0,81.0,35.50,25-30,Hispanic,personalized,True
2,20231010_s0_albert_hammond_act3_wshdw5,20231010,s0,albert_hammond,act3,Loc_32,S7-Cooking,Female,175.0,82.0,26.70,25-30,Hispanic,personalized,True
3,20230825_s1_alejandra_reynolds_act1_5i0rcc,20230825,s1,alejandra_reynolds,act1,Loc_24,S7-Cooking,Male,186.0,83.9,24.25,31-35,Caucasian,personalized,True
4,20231121_s0_alexandria_griffith_act0_945arb,20231121,s0,alexandria_griffith,act0,Loc_41,S7-Cooking,Female,168.0,59.0,21.50,36-40,Caucasian,personalized,True


In [5]:
# Single session — full pipeline in one call
uid = sessions["sequence_uid"].iloc[0]
raw = ngt.load_session(uid, data_root=DATA_ROOT)
meta = sessions.iloc[0].to_dict()

result = ngt.analyze_session(raw, meta=meta, show=True)

# result.df         — preprocessed gaze data
# result.fixations  — fixation table
# result.saccades   — saccade table
# result.summary    — one-row metrics summary
# result.fig        — interactive Plotly figure

In [6]:
print(result.summary.T)

                                                              0
sequence_uid           20230628_s0_adriana_gonzalez_act4_rzxlmo
date                                                   20230628
session_id                                                   s0
fake_name                                      adriana_gonzalez
act_id                                                     act4
location                                                 Loc_10
script                                               S7-Cooking
participant_gender                                         Male
participant_height_cm                                     177.0
participant_weight_kg                                      81.0
participant_bmi                                            25.9
participant_age_group                                     25-30
participant_ethnicity                                 Caucasian
gaze_type                                          personalized
has_gaze_data                           

In [7]:
# Batch analysis — run pipeline across multiple sessions
batch = sessions.iloc[:5]  # first 5 as an example
group = ngt.analyze_sessions(batch, data_root=DATA_ROOT, show=False)

print(group.summaries.shape)
group.summaries

[1/5] 20230628_s0_adriana_gonzalez_act4_rzxlmo
[2/5] 20230929_s1_alan_burns_act4_zch0c7
[3/5] 20231010_s0_albert_hammond_act3_wshdw5
[4/5] 20230825_s1_alejandra_reynolds_act1_5i0rcc
[5/5] 20231121_s0_alexandria_griffith_act0_945arb
(5, 50)


,sequence_uid,date,session_id,fake_name,act_id,location,script,participant_gender,participant_height_cm,participant_weight_kg,...,mean_duration_ms,median_duration_ms,sd_duration_ms,iqr_duration_ms,pct_time_in_fixation,n_saccades,n_artifacts,mean_amplitude_deg,median_amplitude_deg,sd_amplitude_deg
0,20230628_s0_adriana_gonzalez_act4_rzxlmo,20230628,s0,adriana_gonzalez,act4,Loc_10,S7-Cooking,Male,177.0,81.0,...,128.81,99.98,110.34,99.98,31.94,1450,696,2.184,1.314,2.841
1,20230929_s1_alan_burns_act4_zch0c7,20230929,s1,alan_burns,act4,Loc_29,S7-Cooking,Female,151.0,81.0,...,129.96,99.98,92.58,99.98,25.45,1241,703,1.834,1.159,2.022
2,20231010_s0_albert_hammond_act3_wshdw5,20231010,s0,albert_hammond,act3,Loc_32,S7-Cooking,Female,175.0,82.0,...,129.56,99.98,72.65,99.98,21.25,950,693,3.340,2.101,3.673
3,20230825_s1_alejandra_reynolds_act1_5i0rcc,20230825,s1,alejandra_reynolds,act1,Loc_24,S7-Cooking,Male,186.0,83.9,...,125.04,99.98,146.97,99.98,29.06,1233,659,2.531,1.315,3.256
4,20231121_s0_alexandria_griffith_act0_945arb,20231121,s0,alexandria_griffith,act0,Loc_41,S7-Cooking,Female,168.0,59.0,...,128.85,99.98,103.24,0.00,15.23,523,628,1.873,1.222,2.066


In [8]:
# Population density across the batch
ngt.plot_population_density(group.dfs, title="S7-Cooking — Population Gaze Density").show()